### Install library and connect to driver

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
# Cell 1: Mount Google Drive and Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip list | grep unslot

In [ ]:
# Cell 2: Import Libraries and Load Model/Tokenizer
from unsloth import FastLanguageModel
import torch
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

max_seq_length = 12000  # Adjusted to user's requirement
dtype = None  # Auto detection

model_name = "unsloth/Phi-4"

model, tokenizer = FastLanguageModel.from_pretrained( 
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,  # 4-bit quantization for memory efficiency
)

# Prepare for PEFT/LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Efficient for long contexts
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
# Cell 3: Load and Format Dataset
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

dataset_path = "/content/drive/MyDrive/Train/phi4_finetune_dataset_20251221_1901.jsonl"

# Load the full dataset
full_dataset = load_dataset("json", data_files=dataset_path, split="train")

# Split into train and validation (90/10 split)
train_test_split = full_dataset.train_test_split(test_size=0.1)
train_ds = train_test_split["train"]
val_ds = train_test_split["test"]

# Get chat template cho Phi-4
tokenizer = get_chat_template(tokenizer, chat_template="phi-3")

def formatting_prompts_func(example):
    """Format data as messages for chat template.
    User's data format: {"messages": [{"role": "system", "content": ...}, {"role": "user", "content": ...}, {"role": "assistant", "content": ...}]}
    """
    messages = example["messages"]
    # Apply chat template
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

# Apply formatting
train_ds = train_ds.map(formatting_prompts_func, num_proc=2)  # Thêm num_proc để nhanh hơn
val_ds = val_ds.map(formatting_prompts_func, num_proc=2)

# Tính token lengths (input/output/full) TRƯỚC khi remove_columns để giữ thông tin gốc cho thống kê
def calculate_token_lengths(example):
    messages = example["messages"]
    input_content = ""
    output_content = ""
    for msg in messages:
        if msg["role"] in ["system", "user"]:
            input_content += msg["content"] + " "
        elif msg["role"] == "assistant":
            output_content = msg["content"]

    input_tokens = len(tokenizer.encode(input_content.strip()))
    output_tokens = len(tokenizer.encode(output_content.strip()))
    full_tokens = input_tokens + output_tokens  # Gần đúng (không tính special tokens chính xác, nhưng đủ cho filtering)

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "full_tokens": full_tokens
    }

train_ds = train_ds.map(calculate_token_lengths, num_proc=2)
val_ds = val_ds.map(calculate_token_lengths, num_proc=2)

# Sau khi tính token, mới remove_columns (chỉ giữ 'text' cho trainer, nhưng token lengths đã có sẵn)
train_ds = train_ds.remove_columns([col for col in train_ds.column_names if col != 'text' and col not in ['input_tokens', 'output_tokens', 'full_tokens']])
val_ds = val_ds.remove_columns([col for col in val_ds.column_names if col != 'text' and col not in ['input_tokens', 'output_tokens', 'full_tokens']])

print(f"Train size: {len(train_ds)}")
print(f"Val size: {len(val_ds)}")
print("\n" + "="*60)
print("Ví dụ mẫu đã format:")
print("="*60)
print(train_ds[0]['text'][:3000] + "...")

# In thử token lengths của mẫu đầu tiên để kiểm tra
print("\nToken lengths mẫu đầu tiên (train):")
print(f" - Input (system + user): {train_ds[0]['input_tokens']}")
print(f" - Output (assistant): {train_ds[0]['output_tokens']}")
print(f" - Full: {train_ds[0]['full_tokens']}")

In [ ]:
# Cell 4: Measure Token Lengths, Generate Statistics, and Visualize for Reporting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sử dụng trực tiếp các cột token từ train_ds và val_ds (đã có input_tokens, output_tokens, full_tokens)

# Generate statistics
train_stats = pd.DataFrame({
    "input_tokens": train_ds["input_tokens"],
    "output_tokens": train_ds["output_tokens"],
    "full_tokens": train_ds["full_tokens"]
})
val_stats = pd.DataFrame({
    "input_tokens": val_ds["input_tokens"],
    "output_tokens": val_ds["output_tokens"],
    "full_tokens": val_ds["full_tokens"]
})

print("\n📊 Train Token Statistics")
print(train_stats.describe())
print("\n📊 Val Token Statistics")
print(val_stats.describe())

# Save for reporting
train_stats.describe().to_csv("/content/train_token_stats.csv")
val_stats.describe().to_csv("/content/val_token_stats.csv")

# Visualize distributions
plt.figure(figsize=(18, 6))

# Full tokens
plt.subplot(1, 3, 1)
sns.histplot(train_stats['full_tokens'], kde=True, color='blue', label='Train')
sns.histplot(val_stats['full_tokens'], kde=True, color='orange', label='Val')
plt.title('Full Prompt Token Distribution')
plt.legend()

# Input tokens (system + user)
plt.subplot(1, 3, 2)
sns.histplot(train_stats['input_tokens'], kde=True, color='blue', label='Train')
sns.histplot(val_stats['input_tokens'], kde=True, color='orange', label='Val')
plt.title('Input (System + User) Token Distribution')
plt.legend()

# Output tokens (assistant)
plt.subplot(1, 3, 3)
sns.histplot(train_stats['output_tokens'], kde=True, color='blue', label='Train')
sns.histplot(val_stats['output_tokens'], kde=True, color='orange', label='Val')
plt.title('Output (Assistant) Token Distribution')
plt.legend()

plt.tight_layout()
plt.savefig("/content/token_distributions.png")
plt.show()

# Additional visualizations: Box plots for comparison
plt.figure(figsize=(12, 6))
sns.boxplot(data=pd.concat([train_stats.assign(Dataset='Train'), val_stats.assign(Dataset='Val')]),
            x='Dataset', y='full_tokens')
plt.title('Box Plot of Full Token Lengths')
plt.savefig("/content/token_boxplot.png")
plt.show()

In [ ]:
# Cell 5: Filter Datasets by Max Token Length (đã sửa để dùng cột 'full_tokens' sẵn có)
MAX_TOKEN = 12000  # User's max_seq_length

print(f"Filtering data with MAX_TOKEN = {MAX_TOKEN}")
print("="*60)

# Use 'full_tokens' column for filtering (đã tính ở Cell 3)
train_before = len(train_ds)
val_before = len(val_ds)

train_ds_filtered = train_ds.filter(lambda x: x['full_tokens'] <= MAX_TOKEN, num_proc=2)
val_ds_filtered = val_ds.filter(lambda x: x['full_tokens'] <= MAX_TOKEN, num_proc=2)

train_after = len(train_ds_filtered)
val_after = len(val_ds_filtered)

print(f"Train: Before {train_before}, After {train_after}, Removed {(train_before - train_after)/train_before*100:.2f}%")
print(f"Val: Before {val_before}, After {val_after}, Removed {(val_before - val_after)/val_before*100:.2f}%")

# Update datasets
train_ds = train_ds_filtered
val_ds = val_ds_filtered

print(f"\n✅ Ready for training: Train {len(train_ds)}, Val {len(val_ds)}")

# Optional: In thống kê sau filter để báo cáo (sửa để chuyển cột dataset thành list trước DataFrame)
print("\n📊 Post-Filter Token Stats (Train):")
print(pd.DataFrame({"full_tokens": list(train_ds["full_tokens"])}).describe())
print("\n📊 Post-Filter Token Stats (Val):")
print(pd.DataFrame({"full_tokens": list(val_ds["full_tokens"])}).describe())

In [ ]:
# Cell 6: Set Up Trainer with Monitoring and Resume from Checkpoint
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback, TrainerCallback
import os

# Bật expandable_segments để tránh memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Set output_dir to Google Drive to persist checkpoints
output_dir = "/content/drive/MyDrive/Phi4_Finetune_Outputs/outputs_phi4"
os.makedirs(output_dir, exist_ok=True)

training_args = TrainingArguments(
    # Train: Tăng batch size lên cao hơn
    per_device_train_batch_size=112,              # Tăng lên 96 (gần max an toàn trên A100 80GB)
    gradient_accumulation_steps=1,               # Không cần accumulation nữa vì batch đã lớn
    max_steps=60,
    # Eval: Giữ an toàn để tránh OOM ở evaluation
    per_device_eval_batch_size=48,               # Hoặc tăng lên 48 nếu muốn nhanh hơn (test trước)

    warmup_ratio=0.1,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    weight_decay=0.05,
    lr_scheduler_type="cosine",

    # Save thường xuyên để resume dễ dàng
    save_strategy="steps",
    save_steps=10,
    output_dir=output_dir,
    report_to="tensorboard",

    eval_strategy="steps",
    eval_steps=10,
    bf16=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=3,
)

early_stopping = EarlyStoppingCallback(early_stopping_patience=3)

# Callback clear cache trước eval
class ClearCacheCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, **kwargs):
        import torch
        import gc
        torch.cuda.empty_cache()
        gc.collect()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
    dataset_text_field="text",
    packing=False,
    max_seq_length=12000,                   # Nếu OOM, giảm xuống 8192
    callbacks=[early_stopping, ClearCacheCallback()],
)

# Check for last checkpoint and resume if exists
last_checkpoint = None
checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
if checkpoints:
    checkpoints.sort(key=lambda x: int(x.split('-')[-1]))
    last_checkpoint = os.path.join(output_dir, checkpoints[-1])
    print(f"Resuming from checkpoint: {last_checkpoint}")
else:
    print("Không tìm thấy checkpoint nào → Bắt đầu huấn luyện từ đầu.")

trainer_stats = trainer.train(resume_from_checkpoint=last_checkpoint)

print("Training complete.")

In [ ]:
import os

output_dir = "/content/drive/MyDrive/Phi4_Finetune_Outputs/outputs_phi4"
if os.path.exists(output_dir):
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    print("Thư mục outputs_phi4 tồn tại.")
    if checkpoints:
        print("Các checkpoint có sẵn:", checkpoints)
        print("Checkpoint mới nhất:", max(checkpoints, key=lambda x: int(x.split('-')[-1])))
    else:
        print("Không có checkpoint nào trong thư mục!")
else:
    print("Thư mục outputs_phi4 KHÔNG tồn tại trên Drive!")

In [ ]:
# Cell 7: Save Model, Visualize Training Logs, and Save to Drive
# Save model and tokenizer to local first, then copy to Drive
local_model_dir = "/content/fine_tuned_phi4"
model.save_pretrained(local_model_dir)
tokenizer.save_pretrained(local_model_dir)

# Visualize training progress (loss curves)
logs = pd.DataFrame(trainer.state.log_history)

# Filter logs cho train và eval
train_logs = logs[logs['loss'].notna()]  # Các step có train loss
eval_logs = logs[logs['eval_loss'].notna()]  # Các step có eval loss

plt.figure(figsize=(12, 6))
if not train_logs.empty:
    plt.plot(train_logs['step'], train_logs['loss'], label='Train Loss')
if not eval_logs.empty:
    plt.plot(eval_logs['step'], eval_logs['eval_loss'], label='Eval Loss')  # Dùng 'step' cho cả hai
plt.title('Training and Evaluation Loss')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()
local_plot_path = "/content/training_loss_curve.png"
plt.savefig(local_plot_path)
plt.show()

# Save logs cho báo cáo (CSV)
local_logs_path = "/content/training_logs.csv"
logs.to_csv(local_logs_path, index=False)

# For TensorBoard: Chạy trực tiếp trong cell (nếu cần)
%load_ext tensorboard
%tensorboard --logdir {output_dir}

print("Training complete. View TensorBoard for detailed logs.")
print("For inference, use max_new_tokens=2000 as per user's max output tokens.")

# Define a directory in Drive to save outputs
save_dir = "/content/drive/MyDrive/Phi4_Finetune_Outputs/"
!mkdir -p {save_dir}  # Create directory if it doesn't exist

# Copy token statistics CSVs (assuming they exist from previous cells)
!cp /content/train_token_stats.csv {save_dir} || echo "train_token_stats.csv not found, skipping."
!cp /content/val_token_stats.csv {save_dir} || echo "val_token_stats.csv not found, skipping."

# Copy visualization plots (assuming they exist from previous cells)
!cp /content/token_distributions.png {save_dir} || echo "token_distributions.png not found, skipping."
!cp /content/token_boxplot.png {save_dir} || echo "token_boxplot.png not found, skipping."
!cp {local_plot_path} {save_dir}

# Copy model, tokenizer, and outputs (checkpoints already in Drive)
!cp -r {local_model_dir} {save_dir}

# Save trainer logs as JSON
import json
local_json_path = "/content/trainer_logs.json"
with open(local_json_path, "w") as f:
    json.dump(trainer.state.log_history, f)
!cp {local_json_path} {save_dir}
!cp {local_logs_path} {save_dir}

print(f"All important files saved to {save_dir}")

In [ ]:
#  Cell 8: Evaluate Model on test.json
# import json
# from datasets import load_dataset
# from unsloth import FastLanguageModel
# import torch

# # Load model và tokenizer đã fine-tune from Drive (best model after training)
# model_dir = "/content/drive/MyDrive/Phi4_Finetune_Outputs/fine_tuned_phi4"
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_dir,  # Thư mục lưu model
#     max_seq_length=14000,
#     dtype=None,
#     load_in_4bit=True,
# )
# FastLanguageModel.for_inference(model)  # Chế độ inference nhanh

# # Load test dataset (giả định là file JSON chứa list dict)
# test_path = "/content/drive/MyDrive/Train/test.json"

# # Chuyển thành Hugging Face Dataset để dễ xử lý
# test_dataset = load_dataset("json", data_files=test_path, split="train")

# # Hàm generate program từ question + context
# def generate_program(example):
#     # Xây dựng prompt (dựa trên format của bạn: system + user)
#     messages = [
#         {"role": "system", "content": "Bạn là chuyên gia đánh giá kết quả phân tích báo cáo tài chính Việt Nam..."},  # Copy system prompt từ dataset của bạn
#         {"role": "user", "content": f"Câu hỏi: {example['qa']['question']}\n\nContext:\n{'\n'.join(example['pre_text'])}\n\nTable:\n{example['table']}\n\nPost text:\n{'\n'.join(example['post_text'])}"},
#     ]

#     # Apply chat template và generate
#     input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=2000,  # Giới hạn output
#         use_cache=True,
#         temperature=0.0,      # Deterministic
#         do_sample=False,
#     )

#     generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

#     # Trích xuất program (giả định model generate theo format "program: ...")
#     program = generated_text.split("program:")[-1].strip() if "program:" in generated_text else generated_text.strip()

#     return {"generated_program": program}

# # Áp dụng generate cho test dataset bằng loop để tránh multiprocessing issues
# generated_programs = []
# for example in test_dataset:
#     result = generate_program(example)
#     generated_programs.append(result["generated_program"])

# # Thêm cột generated_program vào dataset
# test_dataset = test_dataset.add_column("generated_program", generated_programs)

# # Đánh giá: So sánh generated_program với ground truth program
# correct = 0
# total = len(test_dataset)
# examples = []

# for example in test_dataset:
#     gt_program = example["qa"]["program"]
#     gen_program = example["generated_program"]

#     if gt_program.strip() == gen_program.strip():  # So sánh chính xác
#         correct += 1

#     # Lưu ví dụ sai để debug
#     if gt_program.strip() != gen_program.strip():
#         examples.append({
#             "question": example["qa"]["question"],
#             "ground_truth": gt_program,
#             "generated": gen_program,
#         })

# # In kết quả
# accuracy = correct / total * 100
# print(f"Accuracy trên test set: {accuracy:.2f}% ({correct}/{total} mẫu khớp chính xác)")
# print(f"Số mẫu sai: {len(examples)}")

# # In 5 ví dụ sai (nếu có)
# if examples:
#     print("\nVí dụ sai (5 mẫu đầu):")
#     for ex in examples[:5]:
#         print(f"Question: {ex['question']}")
#         print(f"Ground truth program: {ex['ground_truth']}")
#         print(f"Generated: {ex['generated']}")
#         print("-" * 80)

# # Lưu kết quả đánh giá vào Drive
# eval_save_dir = "/content/drive/MyDrive/Phi4_Finetune_Outputs/"
# eval_path = f"{eval_save_dir}test_evaluation_results.json"
# with open(eval_path, "w", encoding="utf-8") as f:
#     json.dump({
#         "accuracy": accuracy,
#         "total": total,
#         "correct": correct,
#         "errors": examples
#     }, f, ensure_ascii=False, indent=4)

# print(f"Kết quả đánh giá đã lưu vào {eval_path}")